# Phase A — Seed Validation

Re-run the top 3 variants from the Phase A sweep with multiple seeds to
measure loss variance. Determines whether the 1% gap between B (best) and A
(control) is real or within-seed noise.

| Variant | Architecture |
|---------|--------------|
| **A** | B-spline + SiLU base (control) |
| **B** | B-spline + SCReLU base |
| **E** | ReLU-KAN pure basis (arXiv 2406.02075) |

Bullet is non-deterministic by default — weight init uses `rand::rng()` (ThreadRng/OS
entropy) and data shuffle seeds from system time. Each fresh training run uses
different randomness with no code changes needed.

**Plan**: 3 seeds × 3 variants = 9 runs × ~10 min on T4 ≈ 1.5 hours total.

**Runtime**: GPU (T4 or better).

## 1. Install Rust + clone repo

In [ ]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
set -e
if [ -d /content/bullet ]; then
    cd /content/bullet
    git fetch origin
    git reset --hard origin/main
else
    cd /content
    git clone https://github.com/y0sif/bullet.git
    cd bullet
fi
git log -1 --oneline

## 2. Download training data (test77 binpack)

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack from HuggingFace (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Download complete. Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack

## 3. Build the 3 variants (A, B, E)

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
for v in a b e; do
    echo "=== Building kan_variant_${v} ==="
    cargo build --release --example kan_variant_${v} 2>&1 | tail -3
done

## 4. Run 3 seeds per variant (9 runs total)

Each run writes to `/content/variant_<v>_seed<n>_log.txt`. We delete the checkpoints
directory between runs so the trainer doesn't clutter `/content/bullet/checkpoints`
with per-seed subdirectories (we only need the training loss from logs).

In [ ]:
import subprocess, sys, os, shutil

os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

VARIANTS = ["a", "b", "e"]
SEEDS = [1, 2, 3]

# If a previous run stored logs, keep them unless the user wants a clean rerun.
# Set RESET_LOGS=True to force re-running even if a log already exists.
RESET_LOGS = False

for seed in SEEDS:
    for v in VARIANTS:
        log_path = f"/content/variant_{v}_seed{seed}_log.txt"
        if os.path.exists(log_path) and not RESET_LOGS:
            print(f"SKIP {log_path} (already exists; set RESET_LOGS=True to rerun)")
            continue

        # Clean checkpoints so a previous seed's artefacts don't interfere
        ckpt_dir = f"/content/bullet/checkpoints"
        if os.path.isdir(ckpt_dir):
            shutil.rmtree(ckpt_dir, ignore_errors=True)

        print(f"\n{'='*70}\n variant={v} seed={seed}  ->  {log_path}\n{'='*70}")
        cmd = ["cargo", "run", "--release", "--example", f"kan_variant_{v}"]
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            cwd="/content/bullet", text=True, bufsize=1,
        )
        with open(log_path, "w") as log:
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                log.write(line)
        proc.wait()
        print(f"\nExit code: {proc.returncode}")
        if proc.returncode != 0:
            print("FAILED — stopping. Fix the issue before continuing.")
            raise SystemExit(1)

## 5. Aggregate: mean ± stdev per variant, plot, decide

Computes mean and sample stdev of **final** running loss across seeds.
Also plots per-seed loss curves to spot any obviously failed runs.

In [ ]:
import re, statistics
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

def parse_bullet_log(path):
    losses = []
    with open(path) as f:
        for line in f:
            m = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', strip_ansi(line))
            if m:
                losses.append((int(m.group(1)), float(m.group(2))))
    return losses

VARIANTS = ["a", "b", "e"]
LABELS = {
    "a": "A: B-spline + SiLU (control)",
    "b": "B: B-spline + SCReLU",
    "e": "E: ReLU-KAN",
}
SEEDS = [1, 2, 3]

curves = {v: [] for v in VARIANTS}  # v -> list of (seed, losses)
finals = {v: [] for v in VARIANTS}  # v -> list of final loss

for v in VARIANTS:
    for seed in SEEDS:
        path = f"/content/variant_{v}_seed{seed}_log.txt"
        try:
            losses = parse_bullet_log(path)
        except FileNotFoundError:
            print(f"Missing {path}")
            continue
        if not losses:
            print(f"No loss lines in {path}")
            continue
        curves[v].append((seed, losses))
        finals[v].append(losses[-1][1])

# Print the table
print(f"{'Variant':<38}  {'mean':>9}  {'stdev':>8}  {'n':>2}  seeds")
print("-" * 72)
mean_by_v = {}
for v in VARIANTS:
    fs = finals[v]
    if not fs:
        continue
    mu = statistics.mean(fs)
    sd = statistics.stdev(fs) if len(fs) > 1 else float("nan")
    mean_by_v[v] = mu
    seed_str = ", ".join(f"{x:.6f}" for x in fs)
    print(f"{LABELS[v]:<38}  {mu:.6f}  {sd:.2e}  {len(fs):>2}  [{seed_str}]")

# Rank by mean
if mean_by_v:
    ranked = sorted(mean_by_v.items(), key=lambda kv: kv[1])
    print("\nRanking (lower mean = better):")
    for rank, (v, mu) in enumerate(ranked, 1):
        marker = " <-- best" if rank == 1 else ""
        print(f"  {rank}. {LABELS[v]:<38}  {mu:.6f}{marker}")

    # Gap test: if best-vs-second gap > combined stdev, the ordering is likely real
    if len(ranked) >= 2 and len(finals[ranked[0][0]]) > 1 and len(finals[ranked[1][0]]) > 1:
        v_best, mu_best = ranked[0]
        v_second, mu_second = ranked[1]
        gap = mu_second - mu_best
        sd_best = statistics.stdev(finals[v_best])
        sd_second = statistics.stdev(finals[v_second])
        combined = (sd_best**2 + sd_second**2)**0.5
        print(f"\nGap between best ({v_best.upper()}) and second ({v_second.upper()}): {gap:.6f}")
        print(f"Combined stdev: {combined:.2e}  ->  gap/stdev = {gap/combined if combined > 0 else float('inf'):.2f}")
        if combined > 0 and gap > 2 * combined:
            print("  >= 2x stdev: ordering is likely real. Proceed to SPRT.")
        elif combined > 0 and gap > combined:
            print("  >= 1x stdev: weak signal. SPRT will tell you whether it's worth a param sweep.")
        else:
            print("  < 1x stdev: within noise. Treat variants as tied; pick by implementation preference.")

# Plot per-seed curves
fig, ax = plt.subplots(figsize=(11, 7))
colors = {"a": "C0", "b": "C1", "e": "C2"}
for v in VARIANTS:
    for seed, losses in curves[v]:
        alpha = 0.5 if seed != SEEDS[0] else 1.0
        label = LABELS[v] if seed == SEEDS[0] else None
        ax.plot([x[0] for x in losses], [x[1] for x in losses],
                color=colors[v], alpha=alpha, linewidth=2, label=label)
ax.set_xlabel("Superbatch")
ax.set_ylabel("Running loss")
ax.set_title("Phase A Seed Validation — 3 seeds per variant")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/phase_a_seeds.png", dpi=150)
plt.show()

## 6. Save logs + plot to Drive (optional)

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/kanue/phase_a_seeds'
os.makedirs(dest, exist_ok=True)

for v in ["a", "b", "e"]:
    for seed in [1, 2, 3]:
        src = f"/content/variant_{v}_seed{seed}_log.txt"
        if os.path.exists(src):
            shutil.copy(src, dest)

if os.path.exists('/content/phase_a_seeds.png'):
    shutil.copy('/content/phase_a_seeds.png', dest)

print(f"Saved to: {dest}")